# Exploração — GovData (IBGE + INEP)

Este notebook valida o pipeline pós-carga. Rode com o PostgreSQL no ar (`docker compose up -d postgres`) ou, sem banco, direto dos Parquet em `data/processed/`.

Reproduz as mesmas checagens do `powerbi/generate_preview.py`, mas de forma interativa.

In [ ]:
import os, glob, json, sys
import pandas as pd
import psycopg2
from dotenv import load_dotenv
load_dotenv()
print("pandas", pd.__version__)
try:
    import pyspark; print("pyspark", pyspark.__version__)
except ImportError:
    print("pyspark não instalado neste kernel")

In [ ]:
# Tenta PostgreSQL; se falhar, lê dos Parquet
USE_DB = True
try:
    conn = psycopg2.connect(host="localhost", port=5432, dbname=os.getenv("POSTGRES_DB","govdata"), user=os.getenv("POSTGRES_USER","postgres"), password=os.getenv("POSTGRES_PASSWORD","suasenha"), connect_timeout=5)
    print("PostgreSQL OK")
except Exception as e:
    print(f"PostgreSQL indisponível ({e}) — lendo dos Parquet")
    USE_DB = False
    conn = None

In [ ]:
if USE_DB:
    df_estado = pd.read_sql("SELECT * FROM dim_estado ORDER BY id_estado", conn)
else:
    df_estado = pd.read_parquet("../data/processed/estados_processados")
df_estado.head()

In [ ]:
if USE_DB:
    df_ideb = pd.read_sql("SELECT * FROM vw_ideb_medio_regiao_ano ORDER BY ano, nome_regiao", conn)
else:
    # fallback: calcula a média a partir do Parquet (sem a view)
    df = pd.read_parquet("../data/processed/ideb_processado")
    df_ideb = df[df["rede"]=="Pública"].groupby(["id_estado","ano_int"])["ideb_val"].mean().reset_index()
df_ideb.head(10)

In [ ]:
import matplotlib.pyplot as plt
pivot = df_ideb.pivot(index="ano", columns="nome_regiao", values="ideb_medio") if "nome_regiao" in df_ideb.columns else df_ideb.pivot(index="ano_int", columns="id_estado", values="ideb_val")
pivot.plot(marker="o", figsize=(10,6))
plt.title("IDEB médio por região — 2005–2025")
plt.ylabel("IDEB")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
if USE_DB:
    df_pib = pd.read_sql("SELECT uf, valor FROM vw_pib_por_uf_ano WHERE variavel LIKE 'Produto Interno Bruto%' AND ano=(SELECT MAX(ano) FROM vw_pib_por_uf_ano) ORDER BY valor DESC", conn)
    df_pib.plot(x="uf", y="valor", kind="bar", figsize=(12,5), legend=False)
    plt.title("PIB por UF — último ano (R$ mil)")
    plt.ylabel("PIB (R$ mil)")
    plt.show()
    df_pib.head()

## Próximos passos

- Replicar os visuais no Power BI Desktop (`powerbi/REPORT_SPEC.md`).
- Estender com `vw_painel_uf_ano` para dispersão PIB × IDEB.